# `DPR processing` and `Auxip staging` Prefect flows

  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-797
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-798
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-799

## Initialisation

In [2]:
# Imports
from dataclasses import asdict
import os
import os.path as osp

from resources.widget_utils import (
    dpr_proc_radio, deploy_prefect_radio, deploy_prefect, run_prefect_radio, run_prefect, shutdown_checkbox)

from rs_client.ogcapi.dpr_client import DprProcessor
from rs_common.prefect_utils import *
from rs_workflows.flow_utils import  DprProcessIn, Priority, ProcessingMode, WorkflowType
from rs_workflows.init_pi_db_flow import init_pi_database
from rs_workflows.on_demand_processing import dpr_processing, on_demand_cadip_staging

In [3]:
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard_url = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard_url}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [4]:
# Choose prefect deployment method
deploy_prefect_radio

RadioButtons(description='Deploy Prefect flows using:', index=1, options=(('Yaml file and git repository', 'ya…

In [5]:
# Choose prefect flow run method
run_prefect_radio

RadioButtons(description='Run Prefect flows using:', options=(("'prefect deployment run' command line", 'cmd')…

In [6]:
# Choose dpr processor
dpr_proc_radio

RadioButtons(description='DPR processor in this demo:', options=(('MOCKUP', <DprProcessor.MOCKUP: 'mockup'>), …

In [7]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_staging(scale=2)

# Init the processor dask cluster. 
# NOTE: use little resources for now because we only call the task tables.
print(f"** Init Dask cluster for: {dpr_proc_radio.value.name!r} **")
match dpr_proc_radio.value:
    case DprProcessor.MOCKUP:
        init_dask_cluster_mockup(scale=1)
    case DprProcessor.S1L0 | DprProcessor.S3L0:
        init_dask_cluster_l0(scale=1)
    case DprProcessor.S1ARD:
        init_dask_cluster_s1ard(scale=1)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_staging)
display(dask_cluster_eopf)

Auxip service: http://rs-server-adgs:8000/auxip
PRIP service: http://rs-server-prip:8000/prip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Create new dask cluster
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/ecef7d750f3e48579826126e2ae53c11/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| msgpack | 1.1.0  | 1.1.1     | None    |
| tornado | 6.3.3  | 6.5.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-staging' are up: 0/2
Dask workers for 'dask-staging' are up: 2/2
** Init Dask cluster for: 'MOCKUP' **
Connecting to dask gateway for 'dask-eopf-mockup.jgaucher.latest': http://dask-eopf-mockup:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf-mockup.jgaucher.latest': http://localhost:8703/clusters/c1a575082834448abc2fa1f528d22eca/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| msgpack | 1.1.0  | 1.1.1     | None    |
| tornado | 6.3.3  | 6.5.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf-mockup.jgaucher.latest' are up: 0/1
Dask workers for 'dask-eopf-mockup.jgaucher.latest' are up: 1/1


In [8]:
# Get the prefect share bucket folder
share_bucket, _ = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")
s3_code_folder = f"users/{OWNER_ID}/code"

In [ ]:
# Create test collections
INPUT_COLLECTION = "TEST_FLOW_INPUT"
AUXIP_COLLECTION = "TEST_FLOW_AUXIP"
OUTPUT_COLLECTION = "TEST_FLOW_OUTPUT"
for collection in (INPUT_COLLECTION, AUXIP_COLLECTION, OUTPUT_COLLECTION):
  create_test_collection(collection)
  
# Prefect flow environment arguments
flow_env_args = {
  "env": {
    "owner_id": OWNER_ID,
  },
}

# DPR processing input parameters
dpr_process_in = DprProcessIn(
    **flow_env_args, 
    processor_name=dpr_proc_radio.value, 
    processor_version="", # NOTE: is it used ?
    dask_cluster_label=cluster_info_eopf.cluster_label,
    pipeline = "set_me_later",
    unit = "",
    priority = Priority.LOW,
    workflow_type = WorkflowType.ON_DEMAND,
    input_products = {},
    generated_product_to_collection_identifier = {"*": AUXIP_COLLECTION},
    auxiliary_product_to_collection_identifier = {"*": OUTPUT_COLLECTION},
    processing_mode = [ProcessingMode.ALWAYS],
    start_datetime="2023-02-15T11:00:00Z",
    end_datetime="2023-02-16T11:00:00Z",
    satellite=None,
)

ValueError: Exactly one of 'pipeline' or 'unit' must be provided.

## Deploy and run INIT PI DB flow

In [ ]:
# Deploy the Prefect flow
pi_deploy = await deploy_prefect(
    "../../sprint27/init_pi_db_flows.yaml", s3_code_folder, os.environ["PREFECT_WORK_POOL_GENERAL"]
)

In [ ]:
# Run the Prefect flow
await run_prefect(pi_deploy, init_pi_database, flow_env_args)

## Deploy rs-client-libraries Prefect flows

In [ ]:
# Deploy the Prefect flows
dpr_processing_deploy, auxip_deploy, cadip_deploy = await deploy_prefect(
    "./dpr_processing_flow.yaml", s3_code_folder, os.environ["PREFECT_WORK_POOL_EOPF"]
)

## Init the L0 demos

In [ ]:
if dpr_proc_radio.value in (DprProcessor.S1L0, DprProcessor.S3L0):
    print(f"Init demo for: {dpr_proc_radio.value.name!r}")

    if dpr_proc_radio.value == DprProcessor.S1L0:
        dpr_process_in.satellite = "s1"
        cadip_collection = "sgs_sentinel1"
        cadip_session = "S1A_20200105072204051312"
    else:
        dpr_process_in.satellite = "s3"
        cadip_collection = "sgs_sentinel3"
        cadip_session = "S3B_20251010143722593812"

    # Stage a cadip session
    params = {
        **flow_env_args,
        "cadip_collection_identifier": cadip_collection,
        "session_identifier": cadip_session,
        "catalog_collection_identifier": INPUT_COLLECTION,
    }    
    await run_prefect(cadip_deploy, on_demand_cadip_staging, params)

    # Update the input product list of the dpr processing
    dpr_process_in.input_products = {cadip_session: INPUT_COLLECTION}

## Init the S1-ARD demo

<div class="alert alert-block alert-warning">

**NOTE**: for the S1-ARD demo initialization, we need to stage products from the PRIP station. 

This is not implemented yet so for now just do the same init as L0.

**SEE** Pierre's issue: https://gitlab.eopf.copernicus.eu/S1/s1-ard-core/-/issues/21
</div>

In [ ]:
if dpr_proc_radio.value == DprProcessor.S1ARD:
    dpr_process_in.satellite = "s1"    
    cadip_collection = "sgs_sentinel1"
    cadip_session = "S1A_20200105072204051312"
    params = {
        **flow_env_args,
        "cadip_collection_identifier": cadip_collection,
        "session_identifier": cadip_session,
        "catalog_collection_identifier": INPUT_COLLECTION,
    }    
    await run_prefect(cadip_deploy, on_demand_cadip_staging, params)
    dpr_process_in.input_products = {cadip_session: INPUT_COLLECTION}

## Run the DPR processing flow

In [ ]:
# TO BE DISCUSSED: what should we test ?
# For now put the full pipeline depending on the processor.
match dpr_proc_radio.value:
    case DprProcessor.MOCKUP:
        dpr_process_in.pipeline = ""
    case DprProcessor.S1L0:
        dpr_process_in.pipeline = "s1_l0_full"
    case DprProcessor.S3L0:
        dpr_process_in.pipeline = "s3_l0_full"
    case DprProcessor.S1ARD:
        dpr_process_in.pipeline = "s1_ard_full"

# TODO: also run a specific s1ard unit. For L0 we can only run the full pipeline (which is the same as a single unit).

In [ ]:
# Be sure to use the processor chosen by the user
dpr_process_in.processor_name=dpr_proc_radio.value
print(f"Run demo for: {dpr_proc_radio.value.name!r}")

# Run the processor
params = {"dpr_input": asdict(dpr_process_in)}
await run_prefect(dpr_processing_deploy, dpr_processing, params)

In [ ]:
# Get the processing unit list from the last flow run artifacts
# See: https://docs-3.prefect.io/v3/api-ref/rest-api/server/artifacts/read-latest-artifact
response = http_session.get(f"{os.environ['PREFECT_API_URL']}/artifacts/units-list/latest")
response.raise_for_status()
contents = response.json()

# Render the artifact as markdown.
# TO BE DISCUSSED: do we want to save the artifact as markdown 
# or pure json that could be used easier from python code ?
from IPython.display import Markdown
display(Markdown(contents["data"]))


<div class="alert alert-block alert-warning">

TO BE DISCUSSED: in dpr_processing could we save the cql queries as artifacts ? 

Then retrieve them from this demo to run the auxip staging separately from the dpr_processing ?
</div>

## Shutdown the dask clusters

In [ ]:
# Choose to shutdown the dask cluster
shutdown_checkbox

In [ ]:
if shutdown_checkbox.value:
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)
    close_dask_clusters()
# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.